# 9.2 Databricks Multi-Tenant Serving Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/10_production_stories/09.2_databricks_multi_tenant/lab.ipynb)
[![Open In Molab](https://raw.githubusercontent.com/marimo-team/marimo/main/docs/_static/marimo-badge.svg)](https://molab.marimo.io/import/github/harshuljain13/llm-inference-at-scale/blob/master/content/10_production_stories/09.2_databricks_multi_tenant/lab.ipynb)

Analytical experiments exploring Model Units economics, LoRA memory efficiency, MixAttention KV savings, and multi-tenant scheduling.

In [ ]:
# Install minimal dependencies for analytical modeling
import subprocess
subprocess.run(['pip', 'install', '-q', 'matplotlib', 'numpy'], check=True)

import numpy as np  # numerical computation
import matplotlib.pyplot as plt  # visualization of economics and memory

## Experiment 1: Model Units Economics

Compare dedicated vs multi-tenant provisioning cost as tenant count scales.

In [ ]:
def model_units_economics():
    """Model the cost savings of Model Units vs dedicated provisioning."""
    # GPU capacity: tokens/second per A100
    tokens_per_gpu = 4000
    # GPU cost: $/hour per A100
    gpu_cost_per_hour = 2.50
    # Hours per year
    hours_per_year = 8760

    # Tenant profiles: (count, avg_tok/s, peak_tok/s)
    tenant_profiles = [
        (10, 500, 2000),   # high-traffic interactive
        (20, 200, 800),    # moderate batch
        (20, 50, 500),     # low-traffic periodic
    ]

    # Dedicated: provision each tenant for their peak
    dedicated_gpus = sum(
        count * np.ceil(peak / tokens_per_gpu)
        for count, avg, peak in tenant_profiles
    )
    # Average utilization under dedicated provisioning
    total_avg_demand = sum(count * avg for count, avg, peak in tenant_profiles)
    dedicated_util = total_avg_demand / (dedicated_gpus * tokens_per_gpu)

    # Multi-tenant: size for P95 concurrent peak (not sum of all peaks)
    # Statistical multiplexing: P95 peak ~ 55% of sum-of-peaks for 50 tenants
    sum_of_peaks = sum(count * peak for count, avg, peak in tenant_profiles)
    p95_concurrent_peak = sum_of_peaks * 0.55  # empirical: 55% overlap at 50 tenants
    # Add 20% burst buffer + 1 redundancy GPU
    shared_gpus = np.ceil(p95_concurrent_peak / tokens_per_gpu * 1.2) + 1
    shared_util = total_avg_demand / (shared_gpus * tokens_per_gpu)

    # Annual costs
    dedicated_cost = dedicated_gpus * gpu_cost_per_hour * hours_per_year
    shared_cost = shared_gpus * gpu_cost_per_hour * hours_per_year
    savings = dedicated_cost - shared_cost

    # Visualize comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Left: GPU count comparison
    bars = ax1.bar(['Dedicated', 'Model Units'], [dedicated_gpus, shared_gpus],
                   color=['#ffe4e6', '#dcfce7'], edgecolor='#000')
    ax1.set_ylabel('GPUs Required')  # y-axis: hardware needed
    ax1.set_title('Fleet Size: Dedicated vs Model Units')
    # Annotate with values
    ax1.text(0, dedicated_gpus + 0.5, f'{int(dedicated_gpus)}', ha='center', fontsize=14)
    ax1.text(1, shared_gpus + 0.5, f'{int(shared_gpus)}', ha='center', fontsize=14)

    # Right: annual cost comparison
    ax2.bar(['Dedicated', 'Model Units'], [dedicated_cost/1000, shared_cost/1000],
            color=['#ffe4e6', '#dcfce7'], edgecolor='#000')
    ax2.set_ylabel('Annual Cost ($K)')  # y-axis: dollars spent
    ax2.set_title(f'Annual GPU Cost (Savings: ${savings/1000:.0f}K)')
    ax2.text(0, dedicated_cost/1000 + 10, f'${dedicated_cost/1000:.0f}K', ha='center', fontsize=11)
    ax2.text(1, shared_cost/1000 + 10, f'${shared_cost/1000:.0f}K', ha='center', fontsize=11)

    plt.tight_layout()
    plt.show()

    # Print summary
    print(f"Dedicated: {int(dedicated_gpus)} GPUs, {dedicated_util*100:.1f}% avg util")
    print(f"Model Units: {int(shared_gpus)} GPUs, {shared_util*100:.1f}% avg util")
    print(f"Savings: ${savings:,.0f}/year ({savings/dedicated_cost*100:.0f}% reduction)")
    return {'dedicated_gpus': dedicated_gpus, 'shared_gpus': shared_gpus, 'savings': savings}

# Run economics analysis
econ_results = model_units_economics()

## Experiment 2: LoRA Multiplexing Memory Efficiency

Calculate memory savings from multi-LoRA serving vs dedicated model instances.

In [ ]:
def lora_memory_analysis():
    """Compare memory cost: dedicated instances vs multi-LoRA on shared base."""
    # Model parameters
    base_model_gb = 14.0  # Llama 7B in FP16
    # LoRA adapter size: rank * hidden_dim * 2 (A and B matrices) * num_layers * 2 bytes
    ranks = [8, 16, 32, 64]  # LoRA ranks to evaluate
    hidden_dim = 4096  # Llama 7B hidden dimension
    num_layers = 32  # transformer layers
    num_adapters_range = np.arange(1, 101)  # 1 to 100 adapters

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    colors = ['#dbeafe', '#dcfce7', '#f3e8ff', '#fef3c7']  # one per rank

    for i, rank in enumerate(ranks):
        # Adapter size in GB: 2 matrices (A, B) per layer, FP16
        adapter_gb = rank * hidden_dim * 2 * num_layers * 2 / 1e9

        # Memory for dedicated instances: N copies of full model
        dedicated_mem = num_adapters_range * base_model_gb
        # Memory for multi-LoRA: 1 base + N adapters
        shared_mem = base_model_gb + num_adapters_range * adapter_gb

        # Left plot: total memory comparison for rank=16
        if rank == 16:
            ax1.plot(num_adapters_range, dedicated_mem, 'r--',
                     linewidth=2, label='Dedicated (N x base)')
            ax1.plot(num_adapters_range, shared_mem, '-',
                     color='#dcfce7', linewidth=2, label=f'Multi-LoRA (rank={rank})')
            ax1.fill_between(num_adapters_range, shared_mem, dedicated_mem,
                            alpha=0.2, color='#dcfce7')  # shade the savings area

        # Right plot: adapter size by rank
        ax2.bar(i, adapter_gb * 1000, color=colors[i], edgecolor='#000')  # convert to MB

    # Format left plot
    ax1.set_xlabel('Number of Adapters/Tenants')  # x-axis: scale
    ax1.set_ylabel('Total GPU Memory (GB)')  # y-axis: memory required
    ax1.set_title('Memory: Dedicated vs Multi-LoRA (rank=16)')
    ax1.legend()  # comparison legend
    ax1.grid(True, alpha=0.2)

    # Format right plot
    ax2.set_xticks(range(len(ranks)))  # position labels
    ax2.set_xticklabels([f'Rank {r}' for r in ranks])  # label each bar
    ax2.set_ylabel('Adapter Size (MB)')  # y-axis: per-adapter cost
    ax2.set_title('LoRA Adapter Size by Rank (Llama 7B)')
    # Annotate each bar
    for i, rank in enumerate(ranks):
        size_mb = rank * hidden_dim * 2 * num_layers * 2 / 1e6
        ax2.text(i, size_mb + 2, f'{size_mb:.0f} MB', ha='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    # Print savings at 50 adapters, rank 16
    rank16_size = 16 * hidden_dim * 2 * num_layers * 2 / 1e9
    dedicated_50 = 50 * base_model_gb
    shared_50 = base_model_gb + 50 * rank16_size
    print(f"\n50 tenants with rank-16 adapters:")
    print(f"  Dedicated: {dedicated_50:.0f} GB ({dedicated_50/80:.0f} A100s)")
    print(f"  Multi-LoRA: {shared_50:.1f} GB ({shared_50/80:.1f} A100s)")
    print(f"  Savings: {(1 - shared_50/dedicated_50)*100:.0f}%")

# Run LoRA memory analysis
lora_memory_analysis()

## Experiment 3: MixAttention KV Cache Reduction

Quantify KV cache savings from hybrid sliding-window/full attention.

In [ ]:
def mixattention_kv_savings():
    """Model KV cache savings from MixAttention architecture."""
    # Model config: 32 layers, GQA with 8 KV heads, head_dim=128
    num_layers = 32
    kv_heads = 8  # GQA reduces KV heads
    head_dim = 128
    dtype_bytes = 2  # FP16
    # Sliding window size
    window_size = 4096

    # Sequence lengths to evaluate
    seq_lengths = np.array([4096, 8192, 16384, 32768, 65536, 131072])

    # KV cache per token per layer: 2 (K+V) * kv_heads * head_dim * dtype
    bytes_per_token_per_layer = 2 * kv_heads * head_dim * dtype_bytes

    # Standard: all 32 layers store full sequence
    standard_gb = seq_lengths * num_layers * bytes_per_token_per_layer / 1e9

    # MixAttention: 16 sliding (capped at window_size) + 16 full
    sliding_layers = 16  # lower layers with sliding window
    full_layers = 16  # upper layers with full attention
    # Sliding layers: min(seq_len, window_size) tokens cached
    sliding_tokens = np.minimum(seq_lengths, window_size)
    mix_gb = (sliding_tokens * sliding_layers + seq_lengths * full_layers) * bytes_per_token_per_layer / 1e9

    # Savings percentage
    savings_pct = (1 - mix_gb / standard_gb) * 100

    # Plot comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Left: absolute KV cache size
    ax1.plot(seq_lengths / 1024, standard_gb, 'o-', color='#ffe4e6',
             markeredgecolor='#000', linewidth=2, markersize=8, label='Standard (all full attn)')
    ax1.plot(seq_lengths / 1024, mix_gb, 's-', color='#dcfce7',
             markeredgecolor='#000', linewidth=2, markersize=8, label='MixAttention (16 sliding + 16 full)')
    ax1.set_xlabel('Sequence Length (K tokens)')  # x-axis: context size
    ax1.set_ylabel('KV Cache per Request (GB)')  # y-axis: memory consumed
    ax1.set_title('KV Cache Size: Standard vs MixAttention')
    ax1.legend()  # architecture comparison
    ax1.grid(True, alpha=0.2)

    # Right: savings percentage
    ax2.bar([f'{s//1024}K' for s in seq_lengths], savings_pct,
            color='#dbeafe', edgecolor='#000')
    ax2.set_xlabel('Sequence Length')  # x-axis: context sizes
    ax2.set_ylabel('KV Cache Savings (%)')  # y-axis: reduction percentage
    ax2.set_title('MixAttention Memory Savings by Sequence Length')
    ax2.axhline(y=48, color='red', linestyle='--', alpha=0.5, label='~48% at long context')
    ax2.legend()
    # Annotate bars
    for i, v in enumerate(savings_pct):
        ax2.text(i, v + 0.5, f'{v:.0f}%', ha='center', fontsize=9)

    plt.tight_layout()
    plt.show()

    # Print concurrent request capacity
    gpu_mem_for_kv = 55  # GB available after model weights on A100
    std_concurrent_128k = gpu_mem_for_kv / standard_gb[-1]  # at 128K
    mix_concurrent_128k = gpu_mem_for_kv / mix_gb[-1]  # at 128K
    print(f"\nAt 128K context on A100 (55 GB for KV):")
    print(f"  Standard: {std_concurrent_128k:.1f} concurrent requests")
    print(f"  MixAttention: {mix_concurrent_128k:.1f} concurrent requests")
    print(f"  {mix_concurrent_128k/std_concurrent_128k:.1f}x more concurrent capacity")

# Run MixAttention analysis
mixattention_kv_savings()

## Experiment 4: Overbooking Risk Analysis

Simulate how overbooking safety varies with tenant count (law of large numbers).

In [ ]:
def overbooking_risk_simulation():
    """Monte Carlo simulation of overbooking risk vs tenant count."""
    np.random.seed(42)  # reproducible simulation
    # Each tenant: average 100 tok/s, peak 500 tok/s
    # Model traffic as log-normal (heavy-tailed, realistic for web traffic)
    avg_demand = 100  # tokens/sec average per tenant
    peak_demand = 500  # tokens/sec peak per tenant
    # Log-normal parameters derived from avg and peak (P99)
    sigma = 0.8  # log-normal shape (controls burstiness)
    mu = np.log(avg_demand) - sigma**2 / 2  # log-normal location

    tenant_counts = [5, 10, 20, 50, 100, 200]  # number of tenants
    num_simulations = 10000  # Monte Carlo iterations
    # Physical capacity: sized for guaranteed total (N * avg * overbook_ratio)
    overbook_ratio = 1.5  # sell 1.5x physical capacity

    violation_rates = []  # probability of exceeding capacity
    p95_utils = []  # P95 utilization levels

    for n_tenants in tenant_counts:
        # Physical capacity = n_tenants * avg / overbook_ratio
        physical_capacity = n_tenants * avg_demand / overbook_ratio

        # Simulate n_tenants' demand across num_simulations time steps
        # Each tenant's demand is independently log-normal
        demands = np.random.lognormal(mu, sigma, (num_simulations, n_tenants))
        # Total demand across all tenants at each time step
        total_demand = demands.sum(axis=1)

        # Violation: total demand exceeds physical capacity
        violation_rate = (total_demand > physical_capacity).mean()
        violation_rates.append(violation_rate * 100)

        # P95 utilization
        utilization = total_demand / physical_capacity
        p95_utils.append(np.percentile(utilization, 95) * 100)

    # Plot results
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Left: violation probability decreases with tenant count
    ax1.plot(tenant_counts, violation_rates, 'o-', color='#ffe4e6',
             markeredgecolor='#000', linewidth=2, markersize=8)
    ax1.set_xlabel('Number of Tenants')  # x-axis: tenant scale
    ax1.set_ylabel('Capacity Violation Rate (%)')  # y-axis: risk
    ax1.set_title(f'Overbooking Risk vs Tenant Count (ratio={overbook_ratio}x)')
    ax1.grid(True, alpha=0.2)
    ax1.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='1% target')
    ax1.legend()  # risk threshold

    # Right: P95 utilization converges as tenants increase
    ax2.bar([str(n) for n in tenant_counts], p95_utils,
            color='#dbeafe', edgecolor='#000')
    ax2.set_xlabel('Number of Tenants')  # x-axis: scale
    ax2.set_ylabel('P95 Fleet Utilization (%)')  # y-axis: efficiency
    ax2.set_title('Fleet Utilization Stabilizes with More Tenants')
    ax2.axhline(y=85, color='red', linestyle='--', alpha=0.5, label='85% target')
    ax2.legend()

    plt.tight_layout()
    plt.show()

    # Print insights
    print(f"\nLaw of large numbers in action:")
    print(f"  5 tenants: {violation_rates[0]:.1f}% violation rate (risky)")
    print(f"  50 tenants: {violation_rates[3]:.1f}% violation rate (safe)")
    print(f"  200 tenants: {violation_rates[5]:.1f}% violation rate (very safe)")
    print(f"\nOverbooking at 1.5x is safe with 50+ tenants")

# Run overbooking simulation
overbooking_risk_simulation()